## Redis Object Store
Here we can use the Python SDK to utilize a redis object store with an agnet, then save the agent to a config.yaml and run it from there.

In [1]:
import os
import sys

# Import the NeMo-Agent-Toolkit module
module_path = os.path.abspath('../../../src/')
if module_path not in sys.path:
    sys.path.insert(0, module_path)

In [ ]:
user_report_get_description = """
Fetches user diagnostic report from the bucket given a user ID and date.
Args:
    user_id: str: The user ID to fetch the report for.
    date: str | null: The date to fetch the report for. Format: YYYY-MM-DD. If not provided, the latest report will be fetched.
"""  # noqa: E501

user_report_put_description = """
Inserts a new user diagnostic report into the bucket given a user ID and date.
If a report already exists for the (user_id, date) key, do not overwrite it; return a conflict error.
Never delete or replace an existing report as part of this operation.
Args:
    report: str: The report to put into the bucket.
    user_id: str: The user ID to put the report for.
    date: str | null: The date to put the report for. Format: YYYY-MM-DD. If not provided, the report will be named "latest".
"""  # noqa: E501

user_report_update_description = """
Updates a user diagnostic report for the given user_id and date.
If the report does not exist, create it (upsert semantics). Do not delete/recreate existing reports.
The operation should be idempotent for the same (user_id, date, report) inputs.
Args:
    report: str: The report to update in the bucket.
    user_id: str: The user ID to update the report for.
    date: str | null: The date to update the report for. Format: YYYY-MM-DD. If not provided, the report will be named "latest".
"""  # noqa: E501

user_report_delete_description = """
Deletes user diagnostic report from the bucket given a user ID and date.
    If the report does not exist, fail with a not found error.
    Never perform this operation without the explicit intention to delete an existing report, even in the case of an attempt
    to insert a new report which already exists.
    Args:
        user_id: str: The user ID to delete the report for.
        date: str | null: The date to delete the report for. Format: YYYY-MM-DD. If not provided, the report will be named "latest".

"""  # noqa: E501


In [3]:
from nat.data_models.component_ref import ObjectStoreRef
from nat.front_ends.fastapi.fastapi_front_end_config import FastApiFrontEndConfig
from nat.llm.nim_llm import NIMModelConfig
from nat.plugins.redis.object_store import RedisObjectStoreClientConfig
from nat.plugins.redis.object_store import redis_object_store_client
from nat.utils.sdk.nat_agent import NatReactAgent
from nat.utils.sdk.nat_front_end import NatFrontEnd
from nat.utils.sdk.nat_general_configuraton import NatGeneralConfiguration
from nat.utils.sdk.nat_llm import NatLLM
from nat.utils.sdk.nat_object_store import NatObjectStore
from nat.utils.sdk.nat_tool_group import NatToolGroup
from nat_user_report.user_report_tools import UserReportConfig
from nat_user_report.user_report_tools import user_report_group

llm = NatLLM(
    config=NIMModelConfig(model_name="nvdev/meta/llama-3.1-70b-instruct", temperature=0.0, max_tokens=1024),
    name="nim_llm",
)

redis_object_store = NatObjectStore(config=RedisObjectStoreClientConfig(host="localhost",
                                                                        db=0,
                                                                        port=6379,
                                                                        bucket_name="my-bucket"),
                                    function=redis_object_store_client,
                                    name="report_object_store")

user_report_tool_group = NatToolGroup(
    config=UserReportConfig(object_store=ObjectStoreRef(value=redis_object_store.object_store_name),
                            get_description=user_report_get_description,
                            put_description=user_report_put_description,
                            update_description=user_report_update_description,
                            delete_description=user_report_delete_description),
    tool_group=user_report_group,
    name="user_report",
)

front_end_configuration = NatFrontEnd(object_store=ObjectStoreRef(value=redis_object_store.object_store_name),
                                      cors=FastApiFrontEndConfig.CrossOriginResourceSharing(allow_origins=["*"]))

general_agent_configuration = NatGeneralConfiguration(front_end_configuration=front_end_configuration)

agent = NatReactAgent(
    configuration=general_agent_configuration,
    tool_groups=[user_report_tool_group],
    object_stores=[redis_object_store],
    llm=llm,
    verbose=True,
    pass_tool_call_errors_to_agent=True,
    parse_agent_response_max_retries=3,
)

/Users/spastoriza/Documents/Programming/public/nat-fork/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import os
from pathlib import Path

path_to_yaml = Path(os.getcwd(), "config", "config.yaml").resolve()

# Create the config directory if it doesn't exist
if not path_to_yaml.parent.exists():
    os.makedirs(path_to_yaml.parent)

# Save the agent to a config file
agent.save_to_config_file(path_to_yaml)

# Print out the config file content
with open(path_to_yaml) as f:
    print(f.read())

None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


general:
  front_end:
    _type: fastapi
    cors:
      allow_origins:
      - '*'
    object_store: report_object_store

function_groups:
  user_report:
    _type: user_report
    object_store: report_object_store
    get_description: |2

      Fetches user diagnostic report from the bucket given a user ID and date.
      Args:
          user_id: str: The user ID to fetch the report for.
          date: str | null: The date to fetch the report for. Format: YYYY-MM-DD. If not provided, the latest report will be fetched.
    put_description: |2

      Inserts a new user diagnostic report into the bucket given a user ID and date.
      If a report already exists for the (user_id, date) key, do not overwrite it; return a conflict error.
      Never delete or replace an existing report as part of this operation.
      Args:
          report: str: The report to put into the bucket.
          user_id: str: The user ID to put the report for.
          date: str | null: The date to put the re

In [6]:
await agent.prompt('Give me the latest report of user 67890.')

'The latest report of user 67890 is:\n{\n    "user_id": "67890",\n    "timestamp": "2025-04-21T15:40:00Z",\n    "system": {\n      "os": "macOS 14.1",\n      "cpu_usage": "43%",\n      "memory_usage": "8.1 GB / 16 GB",\n      "disk_space": "230 GB free of 512 GB"\n    },\n    "network": {\n      "latency_ms": 95,\n      "packet_loss": "0%",\n      "vpn_connected": true\n    },\n    "errors": [],\n    "recommendations": [\n      "System operating normally",\n      "No action required"\n    ]\n}'